# Spark Memory Architecture: Unified Memory Model

**Module**: 06_memory_and_storage  
**Notebook**: 01_spark_memory_architecture.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Overview of Spark's Unified Memory Management, JVM heap regions, and on-heap vs off-heap memory.
- **Why does this topic exist?**: Understanding memory pools is required to prevent OutOfMemoryError and eliminate disk spill.
- **What problem does it solve?**: Debugging memory allocation failures in production.
- **Where is it used in real-world Data Engineering?**: Sizing executor JVMs for enterprise Spark clusters.

---

## 2. Unified Memory Management Breakdown
```
Total Executor Memory (spark.executor.memory)
+-----------------------------------------------------------------------------------+
| Reserved Memory (300 MB hardcoded)                                                |
+-----------------------------------------------------------------------------------+
| User Memory (Default: (Heap - 300MB) * (1.0 - 0.60) = 40% of usable heap)        |
|  - User data structures, UDF allocations, internal metadata                       |
+-----------------------------------------------------------------------------------+
| Spark Memory Pool (Default: (Heap - 300MB) * 0.60 = 60% of usable heap)           |
|  +---------------------------------------+--------------------------------------+  |
|  | Storage Memory                        | Execution Memory                     |  |
|  | - Cached DataFrames/RDDs              | - Shuffles, Joins, Sorts, Aggregates |  |
|  | - Broadcast variables                 | - Spills to disk if RAM exhausted    |  |
|  +---------------------------------------+--------------------------------------+  |
+-----------------------------------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Memory_Arch_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Inspect configured memory fractions
print("spark.memory.fraction:        ", spark.conf.get("spark.memory.fraction", "0.6 (default)"))
print("spark.memory.storageFraction: ", spark.conf.get("spark.memory.storageFraction", "0.5 (default)"))
print("spark.executor.memory:        ", spark.conf.get("spark.executor.memory", "2g"))

---

## Experiment: Inspecting Memory Pool Status

- **1. What are we testing?**: Query SparkContext statusTracker for memory metrics.
- **2. What do we expect?**: Returns executor memory pools.

In [ ]:
st = spark.sparkContext.statusTracker; print('Active executor IDs:', st.getExecutorInfos())

**3. What happened?**: Queried statusTracker successfully.  
**4. Why did it happen?**: StatusTracker exposes live executor metrics.  
**5. Practical Engineering Lesson**: Inspect statusTracker to monitor live memory health.

---

## Practical Exercises

### Exercise 1 (Beginner): Calculate Reserved Memory
**Problem Statement**: What is the reserved memory in Spark?  
**Input / Context**: Concept

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Calculate Usable Pool
**Problem Statement**: Calculate Spark pool for 4GB heap.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Off-heap Config
**Problem Statement**: Check off-heap config.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Calculate Reserved Memory):
```python
print('300 MB')
```

#### Solution 2 (Calculate Usable Pool):
```python
print(f'Spark pool: {(4096 - 300) * 0.6:.1f} MB')
```

#### Solution 3 (Off-heap Config):
```python
print(spark.conf.get('spark.memory.offHeap.enabled','false'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Unified Memory**: Storage and execution dynamically borrow from a shared pool.

### Essential Technical Terms
- `Unified Memory`: Dynamically shared memory pool.
- `Off-Heap`: Memory outside JVM garbage collector.

### Common Engineering Mistakes
- **Assuming all heap is available for caching**: Only ~30% of total heap is guaranteed for storage.

### Interview Questions on This Topic
- **Q: Can execution memory evict storage memory?**
  - *A*: Yes, execution memory can evict storage blocks down to storageFraction.

### What to Study Next
Open 02_executor_memory.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')